In [ ]:
import os
import sys
import time
import sqlite3
import pandas as pd
import smtplib
from datetime import datetime
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.utils import formataddr

# ======================
# 初期・パス設定
# ======================
start_time = time.time()

try:
    base_dir = os.path.dirname(__file__)
except NameError:
    base_dir = os.getcwd()

# utils/config を使う前提（PROJECT_DIR, GSHEET_NAME, SHEET_NAMEを利用）
sys.path.append(os.path.abspath(os.path.join(base_dir, "..")))
from utils.config import PROJECT_DIR, GSHEET_NAME, SHEET_NAME

# DBパス
user_base = os.path.join(
    os.environ["USERPROFILE"] if os.name == 'nt' else os.path.expanduser("~"),
    "myenv310", PROJECT_DIR
)
db_path = os.path.join(user_base, "db", "output.db")
print(f"[INFO] 使用DB: {db_path}")

# ======================
# 取得するカラム
# ======================
display_columns = [
    "台番号", "機種名", "タイプ",
    "大当り", "確変", "スタート", "最大枚数", "宵越し累計ゲーム数"
]

need_columns = [
    "実行日", "台番号",
    "宵越し最終種別",
    "機種別回転設定値",
    "単発後天井残りゲーム数",
    "確変後天井残りゲーム数",
]

border_cols_all = [f"等価削りあり{n}回転プラマイボーダー残りゲーム数" for n in range(13, 21)]
all_columns = list(dict.fromkeys(display_columns + need_columns + border_cols_all))

# ======================
# DB読み込み（最新日のみ抽出）
# ======================
with sqlite3.connect(db_path) as conn:
    df = pd.read_sql_query(
        f'''SELECT {", ".join([f"[{c}]" for c in all_columns if c])}
            FROM result_table
            ORDER BY ROWID DESC''', conn
    )

if df.empty:
    print("❌ 対象データがありません")
    sys.exit(0)

df["実行日"] = pd.to_datetime(df["実行日"], errors='coerce')
df = df.dropna(subset=["実行日"])
if df.empty:
    print("❌ 実行日が不正で対象データがありません")
    sys.exit(0)

max_date = df["実行日"].dt.date.max()
df_latest = (
    df[df["実行日"].dt.date == max_date]
    .sort_values("実行日", ascending=False)
    .drop_duplicates(subset=["台番号"])
    .copy()
)
print(f"[INFO] 最新日: {max_date} 件数: {len(df_latest)}")

# ======================
# メール送信設定
# ======================
smtp_server = "smtp.gmail.com"
port = 587
sender_email = "straydog12341234@gmail.com"
password = "cyam jmtc pgfr slpx"  # 実運用は os.getenv で
receiver_emails = ["straydog12341234@gmail.com", "selectshop@gmail.com"]

# ✅ メール用リンク列を追加（DBには保存しない）
base_url = f"https://sedoinfinity.xsrv.jp/{PROJECT_DIR}/machines/machine_"
df_latest["リンク"] = df_latest["台番号"].astype(str).apply(
    lambda x: f'<a href="{base_url}{x}.html" target="_blank">リンク</a>'
)

def send_email(subject, html_body):
    msg = MIMEMultipart("alternative")
    msg["From"] = formataddr((GSHEET_NAME, sender_email))
    msg["To"] = ", ".join(receiver_emails)
    msg["Subject"] = subject
    msg.attach(MIMEText(html_body, "html"))
    try:
        with smtplib.SMTP(smtp_server, port) as server:
            server.starttls()
            server.login(sender_email, password)
            server.sendmail(sender_email, receiver_emails, msg.as_string())
        print(f"✅ メール送信完了: {subject}")
    except Exception as e:
        print(f"[ERROR] メール送信失敗: {e}")

# ======================
# ユーティリティ
# ======================
def norm_kind(x):
    s = (str(x) if x is not None else "").strip()
    if s in ("大当り", "大当たり"):
        return "大当り"
    if s in ("確変",):
        return "確変"
    return s

# ======================
# 集約して1通で送るための収集関数
# ======================
def collect_hits(df_base: pd.DataFrame, kind_label: str, remain_col: str):
    """
    種別(kind_label)と残り列(remain_col)に対して、
    回転設定ごとのヒットDataFrameを {n: df_mail} で返す
    """
    out = {}  # n -> df_mail
    df_work = df_base.copy()

    # 数値化
    if remain_col in df_work.columns:
        df_work[remain_col] = pd.to_numeric(df_work[remain_col], errors="coerce")
    else:
        print(f"[WARN] {remain_col} 列が存在しません。{kind_label}分岐はスキップされる可能性があります。")

    if "機種別回転設定値" in df_work.columns:
        df_work["機種別回転設定値"] = pd.to_numeric(df_work["機種別回転設定値"], errors="coerce")
    else:
        print("[WARN] 機種別回転設定値 列が存在しません。分岐はスキップされる可能性があります。")

    # 種別フィルタ
    if "宵越し最終種別" in df_work.columns:
        df_work["__種別"] = df_work["宵越し最終種別"].map(norm_kind)
        df_work = df_work[df_work["__種別"] == kind_label].copy()
    else:
        print("[WARN] 宵越し最終種別 列が存在しません。分岐をスキップします。")
        return out

    # 利用可能なボーダー列
    rotation_to_col = {
        n: f"等価削りあり{n}回転プラマイボーダー残りゲーム数"
        for n in range(13, 21)
    }
    rotation_to_col = {n: col for n, col in rotation_to_col.items() if col in df_work.columns}

    if df_work.empty or not rotation_to_col:
        print(f"[INFO] {kind_label}分岐: 対象データまたは対応カラムなし（スキップ）")
        return out

    for n, border_col in rotation_to_col.items():
        sub = df_work[df_work["機種別回転設定値"] == n].copy()
        if sub.empty:
            continue
        sub[border_col] = pd.to_numeric(sub[border_col], errors="coerce")

        # 条件: remain_col < border_col
        hit = sub[
            sub[remain_col].notna()
            & sub[border_col].notna()
            & (sub[remain_col] < sub[border_col])
        ].copy()

        if hit.empty:
            continue

        # メール表示用に整形
        display_cols = display_columns + ["機種別回転設定値", remain_col, border_col]
        display_cols = [c for c in display_cols if c in hit.columns]
        df_mail = hit.sort_values("台番号", ascending=True)[list(dict.fromkeys(display_cols))]
        out[n] = (df_mail, border_col)

    return out

# ======================
# 収集 → 1通にまとめて送信
# ======================
hits_da = collect_hits(df_latest, "大当り", "単発後天井残りゲーム数")
hits_kk = collect_hits(df_latest, "確変", "確変後天井残りゲーム数")

total_sections = sum(len(x) for x in [hits_da, hits_kk])
total_rows = sum(len(df) for df, _ in list(hits_da.values()) + list(hits_kk.values()))

if total_sections == 0:
    print("[INFO] 条件一致なし。メール送信は行いません。")
else:
    # HTML本文を構築（セクションごとにテーブルを連結）
    parts = []
    style = """
    <style>
      .styled-table{border-collapse:collapse;width:100%}
      .styled-table th,.styled-table td{border:1px solid #ccc;padding:5px;text-align:left}
      .styled-table th{background-color:#f2f2f2}
      h2{margin:0 0 8px 0}
      h3{margin:16px 0 8px 0}
    </style>
    """

    header = f"<h2>{GSHEET_NAME} {SHEET_NAME} 条件一致まとめ ({max_date})</h2>" \
             f"<p>合計セクション: {total_sections} / 合計台数: {total_rows}</p>"
    parts.append(header)

    # 大当りセクション
    if hits_da:
        parts.append("<h3>【大当り→単発残 < 等価削りありn回転プラマイボーダー残】</h3>")
        for n in sorted(hits_da.keys()):
            df_mail, border_col = hits_da[n]
            parts.append(f"<p><b>設定 {n}</b> | 比較列: 単発後天井残りゲーム数 vs {border_col} | 件数: {len(df_mail)}</p>")
            parts.append(df_mail.to_html(index=False, escape=False, border=1, classes="styled-table"))

    # 確変セクション
    if hits_kk:
        parts.append("<h3>【確変→確変残 < 等価削りありn回転プラマイボーダー残】</h3>")
        for n in sorted(hits_kk.keys()):
            df_mail, border_col = hits_kk[n]
            parts.append(f"<p><b>設定 {n}</b> | 比較列: 確変後天井残りゲーム数 vs {border_col} | 件数: {len(df_mail)}</p>")
            parts.append(df_mail.to_html(index=False, escape=False, border=1, classes="styled-table"))

    html_body = f"<html><head>{style}</head><body>{''.join(parts)}</body></html>"
    subject = f"{GSHEET_NAME}{SHEET_NAME} 条件一致まとめ {max_date}（{total_rows}台/{total_sections}枠）"
    send_email(subject, html_body)

# ======================
# 終了
# ======================
end_time = time.time()
print(f"[INFO] スクリプト完了（実行時間: {end_time - start_time:.2f} 秒）")
